In [1]:
import importlib
import peak_gene_utils
importlib.reload(peak_gene_utils)
from peak_gene_utils import *

from utils import *
import squidpy as sq

KeyboardInterrupt: 

## Load data

In [ ]:
# Load cCREs as a DataFrame
ccre_bed = pd.read_csv(
    "mm10-cCREs.bed",
    sep="\t",
    header=None,
    names=["chrom", "start", "end", "ccre_id", "accession", "ccre_type"]
)

In [ ]:
promoter_ccres = ccre_bed[ccre_bed["ccre_type"]=="PLS"]

In [ ]:
atac = sc.read_h5ad("desc_normalized_atac_peaks.h5ad")
expr = sc.read_h5ad("desc_normalized_rna.h5ad")

In [ ]:
sp_merfish = sc.read_h5ad("../../h5ad_files/c_sp_ad.h5ad")

In [ ]:
sp_merfish_names = sp_merfish.var.index.str.lower().tolist()

In [ ]:
data = np.load("cached_data.npz")
idx_atac = data["idx_atac"]
idx_expr = data["idx_expr"]
pdist_ = data["pdist"]

In [ ]:
atac_20000 = atac[:,idx_atac]
expr_2000 = expr[:,idx_expr]

In [ ]:
g_p_similarity = pdist_[20000:,:20000]

In [ ]:
sq.gr.spatial_autocorr(expr_2000, mode="moran")

In [ ]:
sq.gr.spatial_autocorr(atac_20000, mode="moran")

In [ ]:
moran_list = expr_2000.uns["moranI"].head(n=250).index.tolist()

In [ ]:
peak_moran_list = atac_20000.uns["moranI"].head(n=1000).index.tolist()

## Get correlated gene-peak pairs

In [ ]:
# Get only unannotated peaks near genes in moran_list
no_anno_results_by_gene = get_filtered_peak_results_by_gene(
    gene_list=expr_2000.var_names,
    expr=expr_2000,
    atac=atac_20000,
    g_p_similarity=g_p_similarity,
    ccre_bed=ccre_bed,
    top_n=200,
    max_distance=100000,
    keep_annotated=False
)

In [ ]:
# Get only annotated peaks
anno_results_by_gene = get_filtered_peak_results_by_gene(
    gene_list=moran_list,
    expr=expr_2000,
    atac=atac_20000,
    g_p_similarity=g_p_similarity,
    ccre_bed=ccre_bed,
    top_n=300,
    max_distance=100000,
    keep_annotated=True
)

In [ ]:
anno_results_by_peak = get_genes_associated_with_peaks(
    peak_names=peak_moran_list,
    expr=expr_2000,
    atac=atac_20000,
    g_p_similarity=g_p_similarity,
    ccre_bed=ccre_bed,
    keep_annotated=True,   # or False, or None to skip
    max_distance=100000,
    top_n_per_peak=10
)

In [ ]:
no_anno_results_by_peak = get_genes_associated_with_peaks(
    peak_names=peak_moran_list,
    expr=expr_2000,
    atac=atac_20000,
    g_p_similarity=g_p_similarity,
    ccre_bed=ccre_bed,
    keep_annotated=False,   # or False, or None to skip
    max_distance=100_000,
    top_n_per_peak=10
)

## Plot pairs (reduced to 10 pairs for now)

In [ ]:
plot_peak_gene_pairs(
    mapping_dict=no_anno_results_by_peak,
    expr_adata=expr_2000,
    atac_adata=atac_20000,
    mode='peak_to_gene',
    n_pairs=10
)

In [ ]:
plot_peak_gene_pairs(
    mapping_dict=anno_results_by_peak,
    expr_adata=expr_2000,
    atac_adata=atac_20000,
    mode='peak_to_gene',
    n_pairs=10
)

In [ ]:
plot_peak_gene_pairs(
    mapping_dict=anno_results_by_gene,
    expr_adata=expr_2000,
    atac_adata=atac_20000,
    mode='gene_to_peak',
    n_pairs=10
)

In [ ]:
plot_peak_gene_pairs(
    mapping_dict=no_anno_results_by_gene,
    expr_adata=expr_2000,
    atac_adata=atac_20000,
    mode='gene_to_peak',
    n_pairs=10
)

## Peak-Peak correlations

### combine annotated Peaks from both mappings

In [ ]:
# Flatten all peak rows into one DataFrame
p_by_g_df = pd.concat(anno_results_by_gene.values())
p_by_g_df = p_by_g_df[["chrom", "start", "end"]].drop_duplicates()

# Step 1: find promoter-overlapping peaks
by_g_promoter_peaks = get_promoter_overlapping_peaks(p_by_g_df, promoter_ccres=promoter_ccres)

In [ ]:
by_p_list = list(anno_results_by_peak.keys())
g_by_p_df = atac_20000.var.loc[by_p_list, ["chrom", "start", "end"]]  # or your peak dataframe

# Get promoter overlaps
by_p_promoter_peaks = get_promoter_overlapping_peaks(g_by_p_df, promoter_ccres)

In [ ]:
union_peaks = list(set(by_g_promoter_peaks) | set(by_p_promoter_peaks))

In [ ]:
anno_results_by_peak['chr11:24077826-24078325']

In [ ]:
# get peak-peak similarity matrix
p_p_similarity = pdist_[:20000, :20000]

In [ ]:
correlated_peaks = get_correlated_peaks_near_peaks(
    peak_names=union_peaks,
    atac=atac_20000,
    p_p_similarity=p_p_similarity,
    max_distance=50000,
    top_n_per_peak=5
)

In [ ]:
plot_peak_peak_pairs(
    mapping_dict=correlated_peaks,
    atac_adata=atac_20000,
    n_pairs=10,
    size=25
)

In [ ]:
for p in correlated_peaks.keys():
    if p in anno_results_by_peak.keys():
        for g in anno_results_by_peak[p]:
            sq.pl.spatial_scatter(shape=None, adata=expr_2000, color=g)

In [ ]:
atac_2000.obs

## pseudotime analysis with PAGA

In [ ]:
adata = expr.copy()

In [ ]:
adata.X = adata.X.astype("float64")

In [ ]:
sc.pp.recipe_zheng17(adata)

In [ ]:
sc.tl.pca(adata, svd_solver="arpack")

In [ ]:
sc.pp.neighbors(adata, n_neighbors=4, n_pcs=20)
sc.tl.draw_graph(adata)

In [ ]:
sc.pl.draw_graph(adata, color="subclass", legend_loc="on data")

In [ ]:
adata.uns["iroot"]=0

In [ ]:
sc.tl.dpt(adata)

In [ ]:
sq.pl.spatial_scatter(adata, shape=None, color="dpt_pseudotime", size=6)

In [ ]:
adata.obs["root"] = adata.obs["dpt_pseudotime"]==0

In [ ]:
adata.obs